# Category audit

Audits the category columns produced by `data_cleaning.ipynb`, built in
three layers:

1. **Keyword rules** (`category_regex`) -- the primary labeller. Revised four
   times; see the category sections of the cleaning notebook.
2. **Category model** (`category_auto`) -- one logistic regression on the
   title embedding plus the keyword label, trained (out-of-fold) on the
   corrected labels. This is the fully automatic label: **measure accuracy
   on this column.**
3. **Label corrections** (`category`, with `category_source`) -- hand labels
   (user), the gold sample, and Sonnet labels replace the automatic label
   where they exist.

This notebook doesn't recompute anything; it reads the cleaned products file
and `data/category_reference_labels.csv` (all reference labels, with the
current taxonomy re-applied).


In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 80)

p = pd.read_csv('../data/amazon_products_deduped.csv')
refs = pd.read_csv('../data/category_reference_labels.csv')
overridden = p['category_auto'] != p['category_regex']
print(f'{len(p)} products; the model changed the keyword label for {overridden.sum()} ({overridden.mean():.1%})')
p['category_source'].value_counts()


7946 products; the model changed the keyword label for 268 (3.4%)


category_source
pipeline            6974
sonnet_confident     505
user                 255
gold                 159
sonnet_unsure         53
Name: count, dtype: int64

## Accuracy of the automatic labels

Evaluation sets (distinct products; the user's "unsure" verdicts excluded):

- **Gold sample 1** (80): drawn first and used while *tuning* the keyword
  rules, so optimistic.
- **Gold sample 2** (80): drawn after tuning Revisions 1-3, never used to
  write a rule -- the cleanest held-out number, but small (±6 points).
- **Random 250** (user's hand labels): before Revision 4 the pipeline scored
  **87.6%** on it -- the independent estimate at that point. Revision 4's
  rules (phones/tablets, watches, SSDs, car, video gear) were written after
  seeing its errors, so its score now is optimistic.

Reference labels made before Revision 4 have the new rules re-applied, so
all columns are scored against the current taxonomy.


In [2]:
def score(key):
    e = refs[refs['eval_set'] == key].merge(p[['Title', 'category_regex', 'category_auto']], on='Title')
    return pd.Series({'n': len(e),
                      'keyword rules': f"{(e['category_regex'] == e['label_now']).mean():.1%}",
                      'model (category_auto)': f"{(e['category_auto'] == e['label_now']).mean():.1%}"})

pd.DataFrame({name: score(key) for name, key in
              [('gold 1 (tuned on)', 'gold_1'), ('gold 2 (held out)', 'gold_2'), ('random 250 (user)', 'random_250')]})


,gold 1 (tuned on),gold 2 (held out),random 250 (user)
n,79,80,246
keyword rules,94.9%,92.5%,94.3%
model (category_auto),94.9%,93.8%,95.1%


In [3]:
# remaining misses of the automatic label
ev = refs[refs['eval_set'] != ''].dropna(subset=['eval_set']).merge(p[['Title', 'category_regex', 'category_auto']], on='Title')
ev.loc[ev['category_auto'] != ev['label_now'], ['eval_set', 'Title', 'category_regex', 'category_auto', 'label_now']]


,eval_set,Title,category_regex,category_auto,label_now
61,random_250,Humminbird LakeMaster Premium Minnesota V1 Map SD Card for Fish Finders,Marine & GPS,Marine & GPS,Storage & Memory Cards
93,random_250,"Fender Professional Series Instrument Cable, Guitar Cable 10 ft, Guitar Acce...",Cables & Charging,Cables & Charging,Audio
103,random_250,Blackmagic Micro Converter BiDirect SDI/HDMI 3G PSU (CONVBDC/SDI/HDMI03G/PS),Computers & Laptops,Computers & Laptops,Cameras & Photography
108,random_250,"Kasa Smart 3 Way Switch HS210 KIT, Needs Neutral Wire, 2.4GHz Wi-Fi Light Sw...",Networking,Networking,Smart Home & Wearables
138,random_250,"SAMSUNG Galaxy Watch Ultra Trail Band, Smartwatch Replacement Strap for Men ...",Phone & Tablet Accessories,Phone & Tablet Accessories,Smart Home & Wearables
159,random_250,"Corsair Premium 600W PCIe 5.0 12VHPWR PSU Cable RMX, RMX Shift, RMe, HXi, SF...",Computers & Laptops,Computers & Laptops,Cables & Charging
164,random_250,Elgato Master Mount L - Premium Desk Clamp with Pole extendable up to 125cm/...,Audio,Audio,Cameras & Photography
173,random_250,Microsoft Type Cover for Surface Pro - Black (Renewed),Other,Storage & Memory Cards,Computers & Laptops
192,random_250,"HP 230 Wireless Keyboard - Wireless Connection - Low-Profile, Quiet Design -...",Computers & Laptops,Computers & Laptops,Keyboards & Mice
205,random_250,"UGREEN Revodok 7 in 1 USB C Hub Gigabit Ethernet Adapter 4K@60Hz HDMI, 100W ...",Networking,Networking,Cables & Charging


## What the model changed

The better test of the model is reading its changes directly -- especially
on products with no reference label (`category_source == 'pipeline'`),
where nothing else checks it. At introduction: 32 such changes, read as ~28
fixes (TVs out of Computers, switches to Networking, hubs to Cables, iPad
cases to accessories, smartwatches to Wearables), 1 worse (a controller
charging kit to Cables), 2 wrong -> still wrong, 1 debatable. The earlier
"both agree" override read at ~3 fixes per regression.


In [4]:
pd.crosstab(p.loc[overridden, 'category_regex'], p.loc[overridden, 'category_auto'])


category_auto,Audio,Bags & Cases,Cables & Charging,Cameras & Photography,Car Electronics,Computers & Laptops,Keyboards & Mice,Marine & GPS,Networking,Office & School Supplies,Optics,Phone & Tablet Accessories,Power Strips & Surge Protectors,Printers & Ink,Smart Home & Wearables,Storage & Memory Cards,TV & Home Entertainment
category_regex,,,,,,,,,,,,,,,,,
Audio,0,0,2,0,0,0,0,0,1,0,0,0,0,0,0,0,4
Batteries,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
Cables & Charging,5,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
Cameras & Photography,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0
Computers & Laptops,23,0,13,3,0,0,3,0,3,0,2,1,0,0,0,1,4
Gaming,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,4,0
Marine & GPS,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0
Monitors & Displays,12,0,6,0,5,0,0,2,0,0,0,3,0,0,3,0,0
Networking,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,4


In [5]:
p.loc[overridden & (p['category_source'] == 'pipeline'), ['Title', 'category_regex', 'category_auto']]


,Title,category_regex,category_auto
285,"Anker Nano USB C Charger Block, 30W PIQ 3.0 Foldable iPhone Charger Fast Cha...",Phone & Tablet Accessories,Cables & Charging
429,"Roku Streambar SE | 2-in-1 TV Soundbar with Built-in 4K/HD/HDR Streaming, Pr...",Audio,TV & Home Entertainment
629,Edifier R1280T Powered Bookshelf Speakers - 2.0 Active Near Field Studio Mon...,Monitors & Displays,Audio
668,"Creative Pebble V3 Minimalistic 2.0 Desktop Speakers With USB-C Audio, Clear...",Computers & Laptops,Audio
863,LG 77-Inch Class OLED evo C4 Series Smart TV 4K Processor Flat Screen with M...,Computers & Laptops,TV & Home Entertainment
1137,"Westinghouse Roku TV - 24 Inch Smart TV, HD Television with Wi-Fi Connectivi...",Networking,TV & Home Entertainment
1178,"Kasa Smart Plug Power Strip HS300, Surge Protector with 6 Individually Contr...",Smart Home & Wearables,Power Strips & Surge Protectors
1208,TP-Link TL-SF1005P | 5 Port Fast Ethernet PoE Switch | 4 PoE+ 10/100 Mbps Po...,Computers & Laptops,Networking
2207,"Logitech 920002416 MK710 Wireless Desktop Set, Keyboard/Mouse, USB, Black (L...",Computers & Laptops,Keyboards & Mice
2241,Google 45W USB-C Power Charger - Fast-Charging Pixel Phone Charger - Compati...,Phone & Tablet Accessories,Cables & Charging


## What the label corrections changed


In [6]:
corrected = p['category'] != p['category_auto']
print(corrected.sum(), 'products relabelled')
pd.crosstab(p.loc[corrected, 'category_auto'], p.loc[corrected, 'category_source'])


231 products relabelled


category_source,gold,sonnet_confident,sonnet_unsure,user
category_auto,,,,
Audio,0,15,3,2
Bags & Cases,0,2,0,0
Batteries,0,3,1,0
Cables & Charging,1,11,0,1
Cameras & Photography,1,19,5,1
Car Electronics,1,3,0,0
Computers & Laptops,0,40,6,5
Gaming,1,3,1,0
Keyboards & Mice,1,3,0,0


## Review list: within-category price outliers

Tukey fences on log price within the final category. High precision, low
recall -- the most glaring remaining misfits, not a complete error list.


In [7]:
lp = np.log(p['final_price'])
grp = lp.groupby(p['category'])
q1, q3 = grp.transform(lambda s: s.quantile(0.25)), grp.transform(lambda s: s.quantile(0.75))
out = (lp < q1 - 1.5 * (q3 - q1)) | (lp > q3 + 1.5 * (q3 - q1))
print(out.sum(), 'price outliers')
p.loc[out, ['Title', 'category', 'category_source', 'final_price']].sort_values(['category', 'final_price'])


45 price outliers


,Title,category,category_source,final_price
5070,Focal FBATHYSMG Bluetooth® Headphones with Active Noise Cancellation,Audio,sonnet_confident,1499.0000
7496,Motorola Solutions RMU2040 6-Pack Two-Way Radio Digital Non-Display 99 UHF B...,Audio,sonnet_confident,1512.0000
7365,TUMI - Alpha Bravo Expedition Flap Backpack - Large Carry On Backpack - 21.0...,Bags & Cases,sonnet_confident,678.9900
7753,"Energizer EN55-12IT, 12 Volt 55 Amp Hr, Long Lasting Sealed Lead Acid AGM Re...",Batteries,sonnet_confident,179.9900
7887,"Mighty Max Battery ML210-6GC2-6 Volt 210 AH, Dual Terminal, Rechargeable SLA...",Batteries,sonnet_confident,239.9900
2419,"UGREEN Nexode 500W GaN Fast Charging Station - 6 Ports USB-C & USB-A, 240W S...",Cables & Charging,sonnet_confident,199.9900
7522,Motorola 56531 Multi Unit Charger / Cloning Station,Cables & Charging,sonnet_confident,199.9900
3583,"Corsair Dual SSD Mounting Bracket (3.5” Internal Drive Bay to 2.5"", Easy Ins...",Computers & Laptops,sonnet_confident,4.9900
5652,"ARCTIC MX-6 (2 g) - Ultimate Performance Thermal Paste for CPU, Consoles, Gr...",Computers & Laptops,sonnet_confident,5.9900
3929,4PCS Breadboards Kit Include 2PCS 830 Point 2PCS 400 Point Solderless Breadb...,Computers & Laptops,sonnet_unsure,6.5900


## Final category sizes


In [8]:
p['category'].value_counts()


category
Audio                              1205
Computers & Laptops                1152
Printers & Ink                      910
Cameras & Photography               727
Phone & Tablet Accessories          417
Cables & Charging                   378
Monitors & Displays                 332
Storage & Memory Cards              331
Batteries                           321
Networking                          315
Keyboards & Mice                    299
Office & School Supplies            266
Smart Home & Wearables              227
TV & Home Entertainment             205
Phones & Tablets                    186
Gaming                              155
Car Electronics                     146
Power Strips & Surge Protectors     139
Bags & Cases                        122
Marine & GPS                         52
Optics                               48
Other                                13
Name: count, dtype: int64